In [2]:
from collections import Counter

import osmium
import pandas as pd

from isochrone.download import RAW_DATA_DIR, osm_timestamp

OSM_PATH = RAW_DATA_DIR / "oberbayern_2026-11-05.osm.pbf"
print("data up to", osm_timestamp(OSM_PATH))

highway = Counter()
other = {key: Counter() for key in ["foot", "access", "sidewalk", "motorroad", "area", "indoor"]}
ways = osmium.FileProcessor(OSM_PATH, osmium.osm.WAY).with_filter(
    osmium.filter.KeyFilter("highway")
)
for way in ways:
    highway[way.tags["highway"]] += 1
    for key, counter in other.items():
        if key in way.tags:
            counter[way.tags[key]] += 1

for key, counter in other.items():
    print(key, counter.most_common(6))
print(sum(highway.values()), "ways with a highway tag,", len(highway), "values")
pd.Series(highway).sort_values(ascending=False)


data up to 2026-11-03T21:24:23Z
foot [('yes', 33881), ('designated', 32707), ('use_sidepath', 12263), ('no', 4370), ('permissive', 1383), ('private', 746)]
access [('private', 103023), ('no', 5082), ('permissive', 3879), ('yes', 3816), ('customers', 3431), ('destination', 2033)]
sidewalk [('no', 32554), ('both', 18356), ('right', 7358), ('separate', 6563), ('left', 5954), ('yes', 40)]
motorroad [('yes', 2140), ('no', 653)]
area [('yes', 3427), ('no', 40)]
indoor [('yes', 5896), ('no', 101), ('room', 47), ('corridor', 10), ('footway', 3), ('highway', 3)]
984636 ways with a highway tag, 41 values


service              285590
track                198096
footway              159335
path                 105518
residential          100367
unclassified          34269
secondary             23364
tertiary              22099
steps                 16516
primary               10158
living_street          6135
cycleway               4759
motorway               3740
bus_stop               2173
motorway_link          2139
pedestrian             1966
trunk                  1480
primary_link           1119
trunk_link             1095
corridor               1084
secondary_link          841
proposed                824
construction            553
platform                519
tertiary_link           302
bridleway               197
elevator                 96
raceway                  80
rest_area                70
via_ferrata              67
services                 29
busway                   15
road                     13
yes                      13
ladder                    5
disused             

In [3]:
import time

import numpy as np
from scipy.sparse.csgraph import connected_components

from isochrone.osm import build_walking_graph

start = time.perf_counter()
graph = build_walking_graph(OSM_PATH)
print(f"built in {time.perf_counter() - start:.0f} s")

n_nodes = len(graph.node_ids)
print(n_nodes, "nodes,", graph.lengths.nnz // 2, "edges")
print(f"total length {graph.lengths.sum() / 2 / 1000:.0f} km")
print("edge length (m), percentiles 50, 90, 99, 100:")
print(np.percentile(graph.lengths.data, [50, 90, 99, 100]).round(1))

n_components, component = connected_components(graph.lengths, directed=False)
sizes = np.sort(np.bincount(component))[::-1]
print(n_components, f"components; the largest has {sizes[0] / n_nodes:.1%} of the nodes")
print("sizes of the 10 largest:", sizes[:10])
print("components with at most 10 nodes:", (sizes <= 10).sum())


built in 12 s
5156527 nodes, 5450977 edges
total length 130456 km
edge length (m), percentiles 50, 90, 99, 100:
[  15.    50.4  148.6 1845.6]
6854 components; the largest has 98.9% of the nodes
sizes of the 10 largest: [5098093     982     731     643     584     408     311     308     292
     261]
components with at most 10 nodes: 5600


In [4]:
import shapely

from isochrone.osm import read_poly
from isochrone.transfers import load_platforms
from isochrone.walking import build_grid, to_projected

FEED_PATH = RAW_DATA_DIR / "mvv_gtfs_2026-11-01.zip"
POLY_PATH = RAW_DATA_DIR / "oberbayern_2026-11-05.poly"

platforms = load_platforms(FEED_PATH)
grid = build_grid(platforms)

area = read_poly(POLY_PATH)
area_xy = shapely.transform(area, lambda lonlat: np.column_stack(to_projected(*lonlat.T)))
shapely.prepare(area_xy)
print(f"polygon: {len(area.exterior.coords)} vertices, {area_xy.area / 1e6:.0f} km2")

inside = shapely.contains_xy(area_xy, grid[:, 0], grid[:, 1])
print(f"grid cells inside {inside.sum()}, outside {(~inside).sum()} ({(~inside).mean():.1%})")
platform_inside = shapely.contains_xy(area_xy, platforms["x"], platforms["y"])
print("platforms outside", (~platform_inside).sum())
grid = grid[inside]


polygon: 493 vertices, 17960 km2
grid cells inside 724685, outside 201056 (21.7%)
platforms outside 2824


In [5]:
from isochrone.osm import walking_distances

platform_xy = platforms[["x", "y"]].to_numpy()
start = time.perf_counter()
pairs = walking_distances(graph, platform_xy, grid)
print(f"computed in {time.perf_counter() - start:.0f} s: {len(pairs)} platform-cell pairs")

pairs["air_m"] = np.hypot(*(platform_xy[pairs["source"]] - grid[pairs["target"]]).T)
far = pairs[pairs["air_m"] > 300]
ratio = far["distance_m"] / far["air_m"]
print("network / straight line (pairs over 300 m), percentiles 10, 25, 50, 75, 90:")
print(np.percentile(ratio, [10, 25, 50, 75, 90]).round(2))

has_platform = np.zeros(len(grid), dtype=bool)
has_platform[pairs["target"]] = True
print(f"cells with a platform within 1300 m on the network: {has_platform.sum()}"
      f" of {len(grid)} ({has_platform.mean():.1%})")
print("platforms per cell, percentiles 10, 50, 90:",
      np.percentile(pairs.groupby("target").size(), [10, 50, 90]))


computed in 174 s: 4205608 platform-cell pairs
network / straight line (pairs over 300 m), percentiles 10, 25, 50, 75, 90:
[1.13 1.2  1.31 1.46 1.69]
cells with a platform within 1300 m on the network: 612691 of 724685 (84.5%)
platforms per cell, percentiles 10, 50, 90: [ 2.  4. 14.]


In [6]:
import datetime as dt

import pandas as pd

from isochrone.connections import build_connections
from isochrone.csa import build_network, latest_departures
from isochrone.gtfs import read_gtfs_table
from isochrone.maps import band_polygons, isochrone_map
from isochrone.osm import find_nearby_platforms_on_network
from isochrone.transfers import build_transfers
from isochrone.walking import latest_departures_from_points, walking_time_s

connections = build_connections(FEED_PATH, dt.date(2026, 12, 9))
network = build_network(connections, build_transfers(FEED_PATH))
arrival = 9 * 3600

stops = read_gtfs_table(FEED_PATH, "stops", usecols=["stop_id", "stop_lat", "stop_lon"])
garching = stops.set_index("stop_id").loc["de:09184:460"]
garching_lonlat = (float(garching["stop_lon"]), float(garching["stop_lat"]))
dest_xy = np.column_stack(to_projected(*garching_lonlat))

egress = find_nearby_platforms_on_network(graph, dest_xy, platforms)
targets = dict(zip(egress["stop_id"], (arrival - egress["walk_s"]).tolist()))
departures = latest_departures(network, targets)

start = time.perf_counter()
access = find_nearby_platforms_on_network(graph, grid, platforms)
print(f"access pairs: {len(access)} in {time.perf_counter() - start:.0f} s")

direct = walking_distances(graph, dest_xy, grid, max_distance_m=7200.0)
direct_walk_s = np.full(len(grid), np.nan)
direct_walk_s[direct["target"]] = walking_time_s(direct["distance_m"], detour_factor=1.0)

latest_osm = latest_departures_from_points(access, departures, direct_walk_s, arrival)
minutes_osm = (arrival - latest_osm) / 60
print(f"cells reaching Garching within 2 h: {np.isfinite(minutes_osm).sum()} of {len(grid)}")

bands = band_polygons(grid, minutes_osm)
display(bands[["low_min", "high_min", "n_cells"]])
map_path = RAW_DATA_DIR.parent / "maps" / "garching_2026-12-09_0900_osm.html"
isochrone_map(bands, garching_lonlat).save(map_path)
print(map_path)


access pairs: 4205608 in 174 s
cells reaching Garching within 2 h: 226230 of 724685


,low_min,high_min,n_cells
0,0.0,5.0,16
1,5.0,10.0,81
2,10.0,15.0,213
3,15.0,20.0,430
4,20.0,25.0,771
5,25.0,30.0,1334
6,30.0,35.0,2224
7,35.0,40.0,3240
8,40.0,45.0,3793
9,45.0,50.0,4912


/Users/francescocodeca/Desktop/project_maps/data/maps/garching_2026-12-09_0900_osm.html


In [7]:
from isochrone.maps import DEFAULT_BANDS_MIN
from isochrone.walking import find_nearby_platforms

egress_air = find_nearby_platforms(dest_xy, platforms)
targets_air = dict(zip(egress_air["stop_id"], (arrival - egress_air["walk_s"]).tolist()))
departures_air = latest_departures(network, targets_air)
access_air = find_nearby_platforms(grid, platforms)
direct_air_s = walking_time_s(np.hypot(*(grid - dest_xy).T))
latest_air = latest_departures_from_points(access_air, departures_air, direct_air_s, arrival)
minutes_air = (arrival - latest_air) / 60

reach_air, reach_osm = np.isfinite(minutes_air), np.isfinite(minutes_osm)
both = reach_air & reach_osm
print(f"reachable within 2 h: both {both.sum()}, only straight line {(reach_air & ~both).sum()},"
      f" only OSM {(reach_osm & ~both).sum()}")

diff = minutes_osm[both] - minutes_air[both]
print("OSM minus straight line (min), percentiles 5, 25, 50, 75, 95:")
print(np.percentile(diff, [5, 25, 50, 75, 95]).round(1))
print(f"OSM slower by more than 5 min: {(diff > 5).mean():.1%},"
      f" faster by more than 5 min: {(diff < -5).mean():.1%}")

limits = np.asarray(DEFAULT_BANDS_MIN)
band_air = np.maximum(np.searchsorted(limits, minutes_air[both]), 1)
band_osm = np.maximum(np.searchsorted(limits, minutes_osm[both]), 1)
print("band shift (OSM band minus straight-line band, clipped to ±2), share of cells:")
shift = pd.Series(np.clip(band_osm - band_air, -2, 2))
print(shift.value_counts(normalize=True).sort_index().round(3))


reachable within 2 h: both 222687, only straight line 37249, only OSM 3543
OSM minus straight line (min), percentiles 5, 25, 50, 75, 95:
[-2.  -0.3  0.7  2.4  7.3]
OSM slower by more than 5 min: 9.6%, faster by more than 5 min: 0.6%
band shift (OSM band minus straight-line band, clipped to ±2), share of cells:
-2    0.001
-1    0.026
 0    0.829
 1    0.129
 2    0.015
Name: proportion, dtype: float64


In [7]:
import r5py
from pyproj import Transformer

from isochrone.validation import r5py_latest_departures, write_feed_without_empty_tables
from isochrone.walking import PROJECTED_CRS

PROCESSED_DIR = RAW_DATA_DIR.parent / "processed"
r5_feed = PROCESSED_DIR / "mvv_gtfs_2026-11-01_r5.zip"
print("tables left out:", write_feed_without_empty_tables(FEED_PATH, r5_feed))
r5_network = r5py.TransportNetwork(OSM_PATH, [r5_feed])

rng = np.random.default_rng(0)
reached = np.flatnonzero(np.isfinite(minutes_osm))
unreached = np.flatnonzero(~np.isfinite(minutes_osm))
sample = np.concatenate(
    [rng.choice(reached, 1000, replace=False), rng.choice(unreached, 250, replace=False)]
)
to_lonlat = Transformer.from_crs(PROJECTED_CRS, "EPSG:4326", always_xy=True)
sample_lonlat = np.column_stack(to_lonlat.transform(grid[sample, 0], grid[sample, 1]))

start = time.perf_counter()
latest_r5 = r5py_latest_departures(
    r5_network, sample_lonlat, garching_lonlat, dt.date(2026, 12, 9), arrival
)
print(f"r5py: {time.perf_counter() - start:.0f} s, reached {np.isfinite(latest_r5).sum()}")
np.savez(PROCESSED_DIR / "r5py_garching_2026-12-09_0900.npz", sample=sample, latest_r5=latest_r5)


tables left out: ['shapes.txt']


/opt/anaconda3/envs/isochrone/lib/python3.12/site-packages/r5py/r5/regional_task.py:250: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
/opt/anaconda3/envs/isochrone/lib/python3.12/site-packages/r5py/r5/regional_task.py:250: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
/opt/anaconda3/envs/isochrone/lib/python3.12/site-packages/r5py/r5/regional_task.py:250: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
/opt/anaconda3/envs/isochrone/lib/python3.12/site-packages/r5py/r5/regional_task.py:250: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
/opt/anaconda3/envs/isochrone/lib/python3.12/site-packages/r5py/r5/regional_task.py:250: RuntimeWarning: The pro

r5py: 2473 s, reached 971


In [ ]:
PROCESSED_DIR = RAW_DATA_DIR.parent / "processed"
saved = np.load(PROCESSED_DIR / "r5py_garching_2026-12-09_0900.npz")
sample, latest_r5 = saved["sample"], saved["latest_r5"]


In [9]:
ours_sample = latest_osm[sample]
dist_sample = np.hypot(*(grid[sample] - dest_xy).T)
reach_ours, reach_r5 = np.isfinite(ours_sample), np.isfinite(latest_r5)
anomaly = reach_r5 & (latest_r5 == arrival) & (dist_sample > 500)
print(f"R5 travel time 0 more than 500 m from D (excluded): {anomaly.sum()}")
both = reach_ours & reach_r5 & ~anomaly
print(f"reachable within 2 h: both {both.sum()}, only ours {(reach_ours & ~reach_r5).sum()},"
      f" only r5py {(reach_r5 & ~reach_ours & ~anomaly).sum()},"
      f" neither {(~reach_ours & ~reach_r5).sum()}")

diff = (np.floor(ours_sample[both] / 60) * 60 - latest_r5[both]) / 60
print("ours minus r5py, latest departure (min), percentiles 5, 10, 25, 50, 75, 90, 95:")
print(np.percentile(diff, [5, 10, 25, 50, 75, 90, 95]))
print(pd.Series(np.clip(diff, -6, 6)).value_counts(normalize=True).sort_index().round(3))


R5 travel time 0 more than 500 m from D (excluded): 1
reachable within 2 h: both 952, only ours 47, only r5py 18, neither 232
ours minus r5py, latest departure (min), percentiles 5, 10, 25, 50, 75, 90, 95:
[-8. -3.  0.  1.  1.  2.  3.]
-6.0    0.061
-5.0    0.008
-4.0    0.014
-3.0    0.019
-2.0    0.024
-1.0    0.029
 0.0    0.099
 1.0    0.599
 2.0    0.092
 3.0    0.014
 4.0    0.009
 5.0    0.005
 6.0    0.026
Name: proportion, dtype: float64


In [10]:
names = read_gtfs_table(FEED_PATH, "stops", usecols=["stop_id", "stop_name"])
stop_name = names.set_index("stop_id")["stop_name"]

cells = sample[both]
first = access[access["point"].isin(cells)].copy()
first["via_s"] = first["stop_id"].map(departures) - first["walk_s"]
first = first.dropna(subset="via_s")
first = first.loc[first.groupby("point")["via_s"].idxmax()].set_index("point")

tails = pd.DataFrame(
    {
        "diff_min": diff,
        "dist_km": dist_sample[both] / 1000,
        "ours_min": (arrival - ours_sample[both]) / 60,
        "r5_min": (arrival - latest_r5[both]) / 60,
    },
    index=pd.Index(cells, name="cell"),
)
tails["first_stop"] = first["stop_id"].map(stop_name)
tails["access_m"] = first["distance_m"]
tails["walk_only"] = arrival - direct_walk_s[cells] >= ours_sample[both]

side = np.sign(tails["diff_min"]).where(tails["diff_min"].abs() > 5, 0).rename("side")
print(
    tails.groupby(side)
    .agg(
        n=("diff_min", "size"),
        dist_km=("dist_km", "median"),
        access_m=("access_m", "median"),
        walk_only=("walk_only", "sum"),
    )
    .round(1)
)
tails = tails[side != 0].sort_values("diff_min")
display(tails.head(8).round(1), tails.tail(8).round(1))


        n  dist_km  access_m  walk_only
side                                   
-1.0   58     19.2    1019.8          5
 0.0  869     25.4     709.7          1
 1.0   25     21.6     570.3          0


,diff_min,dist_km,ours_min,r5_min,first_stop,access_m,walk_only
cell,,,,,,,
498884,-60.0,5.8,117.7,58.0,NaN,NaN,True
579381,-39.0,4.2,83.0,45.0,NaN,NaN,True
635233,-35.0,13.4,119.5,85.0,"Gesseltshausen, Abzw.",1228.3,False
475928,-32.0,24.3,119.2,88.0,"Gröbenzell, Westendstraße",1148.2,False
581728,-31.0,5.0,98.9,68.0,NaN,NaN,True
461998,-28.0,21.0,117.1,90.0,Osterangerstraße,937.9,False
560805,-26.0,3.3,99.0,73.0,NaN,NaN,True
451593,-26.0,21.1,110.0,85.0,Langwied Ortsmitte,930.6,False


,diff_min,dist_km,ours_min,r5_min,first_stop,access_m,walk_only
cell,,,,,,,
604403,13.0,17.0,89.9,103.0,"Niederding, Ost",775.4,False
441669,13.0,26.6,101.5,115.0,"Hohenlinden, Lagerhausstraße",330.2,False
295733,14.0,33.6,100.0,115.0,"Wangen, Olympiastraße",1142.7,False
545525,24.0,21.6,86.2,111.0,Bachern Gleis 1,908.0,False
520024,27.0,30.6,88.5,116.0,Weiherhaus,268.2,False
680420,30.0,23.8,86.0,116.0,"Niederhummel, Feuerwehrhaus",299.4,False
618687,33.0,12.1,67.0,100.0,"Flughafen, Besucherpark Gleis 1",1199.1,False
689144,36.0,27.2,83.7,120.0,Thonstetten,461.5,False


In [11]:
slower = tails[tails["diff_min"] < 0].copy()
slower_cells = slower.index.to_numpy()
access_far = find_nearby_platforms_on_network(
    graph, grid[slower_cells], platforms, max_distance_m=7200.0
)
access_far["point"] = slower_cells[access_far["point"].to_numpy()]
latest_far = latest_departures_from_points(access_far, departures, direct_walk_s, arrival)
r5_by_cell = pd.Series(latest_r5, index=sample)
slower["diff_far_min"] = (
    np.floor(latest_far[slower_cells] / 60) * 60 - r5_by_cell[slower_cells]
) / 60

access_far["via_s"] = access_far["stop_id"].map(departures) - access_far["walk_s"]
access_far = access_far.dropna(subset="via_s")
best_far = access_far.loc[access_far.groupby("point")["via_s"].idxmax()].set_index("point")
slower["access_far_m"] = best_far["distance_m"]

explained = slower["diff_far_min"].abs() <= 5
print(f"slower by more than 5 min: {len(slower)}; within ±5 min with 7200 m: {explained.sum()}")
print("access walk of the new best (m), explained cells, percentiles 10, 50, 90:")
print(np.percentile(slower.loc[explained, "access_far_m"], [10, 50, 90]).round())
display(slower.loc[~explained, ["diff_min", "diff_far_min", "first_stop", "access_m",
                                "access_far_m"]].round(1))


slower by more than 5 min: 58; within ±5 min with 7200 m: 43
access walk of the new best (m), explained cells, percentiles 10, 50, 90:
[1350. 1625. 2170.]


,diff_min,diff_far_min,first_stop,access_m,access_far_m
cell,,,,,
560805,-26.0,-16.0,NaN,NaN,2336.9
451593,-26.0,-6.0,Langwied Ortsmitte,930.6,2048.0
472640,-20.0,-9.0,"Markt Schwaben,Graf-Siegh.-Weg",444.7,1698.9
469960,-19.0,-7.0,"Markt Schwaben,Graf-Siegh.-Weg",841.6,2095.8
466259,-19.0,-8.0,"Markt Schwaben,Graf-Siegh.-Weg",1239.3,2479.8
600261,-17.0,-17.0,"Hallbergmoos, Bgm.-Gruber-Str.",670.1,670.1
451899,-13.0,-13.0,"Anzing, Gewerbegebiet Nord",1059.2,1938.6
474471,-12.0,-12.0,"Forstern, Hauptstraße",672.9,672.9
478024,-11.0,-11.0,"Forstern, Gewerbegebiet",998.3,998.3


In [12]:
from isochrone.csa import journey, scan_latest_departures

scan = scan_latest_departures(network, targets)
trips = read_gtfs_table(FEED_PATH, "trips", usecols=["trip_id", "route_id"])
routes = read_gtfs_table(FEED_PATH, "routes", usecols=["route_id", "route_short_name"])
line = trips.merge(routes, on="route_id").set_index("trip_id")["route_short_name"]


def clock(seconds):
    """Format seconds after midnight as HH:MM:SS."""
    s = int(seconds)
    return f"{s // 3600:02d}:{s // 60 % 60:02d}:{s % 60:02d}"


def show_journey(cell):
    """Print our journey from a grid cell, with stop names, clock times and lines."""
    start = first.loc[cell]
    legs = journey(network, scan, start["stop_id"])
    print(
        f"cell {cell}: ours {tails.loc[cell, 'ours_min']:.1f} min,"
        f" r5py {tails.loc[cell, 'r5_min']:.0f} min,"
        f" leave {clock(legs['dep_time'].iloc[0] - start['walk_s'])},"
        f" walk {start['distance_m']:.0f} m to the first stop"
    )
    legs["line"] = legs["trip_id"].map(line)
    for column in ["from_stop", "to_stop"]:
        legs[column] = legs[column].map(stop_name)
    for column in ["dep_time", "arr_time"]:
        legs[column] = legs[column].map(clock)
    display(legs.drop(columns="trip_id"))


for cell in [600261, 452863, 474471, 618687, 689144, 545525]:
    show_journey(cell)


cell 600261: ours 69.2 min, r5py 53 min, leave 07:50:49, walk 670 m to the first stop


,kind,from_stop,to_stop,dep_time,arr_time,line
0,ride,"Hallbergmoos, Bgm.-Gruber-Str.",Neufahrn,08:02:00,08:21:00,691
1,ride,Neufahrn,"Garch.,Forschungsz.",08:33:00,08:49:00,690


cell 452863: ours 75.8 min, r5py 66 min, leave 07:44:10, walk 649 m to the first stop


,kind,from_stop,to_stop,dep_time,arr_time,line
0,ride,"Kirchheim, Lindenviertel","Aschheim, Erdinger Straße",07:55:00,08:02:00,263
1,walk,"Aschheim, Erdinger Straße","Aschheim, Mitte",08:02:00,08:05:22,NaN
2,ride,"Aschheim, Mitte","Hochbrück, Voithstraße",08:15:00,08:41:00,X202
3,walk,"Hochbrück, Voithstraße","Hochbrück, Voithstraße",08:41:00,08:43:28,NaN
4,ride,"Hochbrück, Voithstraße",Garching-Hochbrück,08:46:00,08:50:00,X201
5,walk,Garching-Hochbrück,Garching-Hochbrück Gleis 1,08:50:00,08:52:03,NaN
6,ride,Garching-Hochbrück Gleis 1,"Garching, Forschungszentrum Gleis 1",08:54:30,08:59:20,U6


cell 474471: ours 113.2 min, r5py 102 min, leave 07:06:47, walk 673 m to the first stop


,kind,from_stop,to_stop,dep_time,arr_time,line
0,ride,"Forstern, Hauptstraße","Hörlkofen, Ort",07:18:00,07:30:00,445V
1,walk,"Hörlkofen, Ort",Hörlkofen Gleis 2,07:30:00,07:39:59,NaN
2,ride,Hörlkofen Gleis 2,Ostbahnhof Gleis 11,07:40:00,08:05:00,RB 40
3,walk,Ostbahnhof Gleis 11,Ostbahnhof U-Bahn Gleis 1,08:05:00,08:10:30,NaN
4,ride,Ostbahnhof U-Bahn Gleis 1,Odeonsplatz U4/U5 Gleis 1 Ri Karlsplatz (Stachus),08:23:20,08:27:50,U5
5,walk,Odeonsplatz U4/U5 Gleis 1 Ri Karlsplatz (Stachus),Odeonsplatz U3/U6 Gleis 1 Ri Universität,08:27:50,08:32:14,NaN
6,ride,Odeonsplatz U3/U6 Gleis 1 Ri Universität,"Garching, Forschungszentrum Gleis 1",08:35:50,08:59:20,U6


... (2 outputs left out)

cell 689144: ours 83.7 min, r5py 120 min, leave 07:36:18, walk 462 m to the first stop


,kind,from_stop,to_stop,dep_time,arr_time,line
0,ride,Thonstetten,"Moosburg, Bahnhof",07:44:00,07:49:00,688
1,walk,"Moosburg, Bahnhof",Moosburg Gleis 2/3,07:49:00,07:51:30,NaN
2,ride,Moosburg Gleis 2/3,Unterschleißheim Gleis 2,07:52:00,08:17:00,RB 33
3,walk,Unterschleißheim Gleis 2,Unterschleißheim (S) Ost,08:17:00,08:21:55,NaN
4,ride,Unterschleißheim (S) Ost,Garching-Hochbrück,08:23:00,08:47:00,219
5,walk,Garching-Hochbrück,Garching-Hochbrück Gleis 1,08:47:00,08:49:21,NaN
6,ride,Garching-Hochbrück Gleis 1,"Garching, Forschungszentrum Gleis 1",08:54:30,08:59:20,U6


cell 545525: ours 86.2 min, r5py 111 min, leave 07:33:51, walk 908 m to the first stop


,kind,from_stop,to_stop,dep_time,arr_time,line
0,ride,Bachern Gleis 1,Marienplatz Gleis 1,07:49:00,08:24:00,S2
1,walk,Marienplatz Gleis 1,Marienplatz U-Bahn Gleis 1 Ri Odeonsplatz,08:24:00,08:27:57,NaN
2,ride,Marienplatz U-Bahn Gleis 1 Ri Odeonsplatz,"Garching, Forschungszentrum Gleis 1",08:34:20,08:59:20,U6
